# Class day: laptop ↔ Cutebot Bluetooth link test

Run the cells **top to bottom**, one at a time (Shift+Enter). Each step says what you should see.
The connection to the cars stays open between cells, and the results collect into a table at the end for the report (§3.2).

**Rehearse at home:** set `LINK = "sim"` in the settings cell and run everything with simulated cars.

| Step | What it checks | Cars needed |
|---|---|---|
| 1 | Connect + ID assignment | 1 |
| 2 | Ping latency | 1 |
| 3 | Wheels move the right way; safety stop | 1, on a cup |
| 4 | Drop → automatic reconnect | 1 |
| 5 | All three cars, right car ↔ right robot ID | 3, on cups |
| 6 | Speed calibration | 1, on the floor |
| 7 | Latency with the phone streaming (real load) | 3 + phone |
| 8 | Save results, disconnect | |

## Before class (do this the night before)

1. **Compile the firmware** at [makecode.microbit.org](https://makecode.microbit.org): New Project
   * **Extensions** → add **bluetooth** (accept removing radio) → add **Cutebot** (ELECFREAKS)
   * Gear → **Project Settings** → **No Pairing Required**
   * **JavaScript** tab → replace everything with `firmware/cutebot_ble.ts` → check there are no red underlines → **Download** the `.hex`
   * If `cuteBot.motors`/`cuteBot.stopcar` is underlined: Blocks → Cutebot drawer → drag out the wheel-speed block → switch to JavaScript to see its real name.
2. **Pack:** micro-USB cable, spare batteries, masking tape + pen, tape measure, 3 cups/boxes, phone + tripod + markers.

## In class, before running anything

* Turn **off** other Bluetooth devices on the laptop (headphones, controllers).
* **Don't pair** micro:bits in Windows Bluetooth settings; *remove* any listed there.
* micro:bits must be **V2** (gold logo on the front).
* Flash the `.hex` by dragging it onto the **MICROBIT** drive. Switch the car on: it shows a ◇ diamond. Press **B** to scroll its 5-letter **name**, and write it on tape on the car.
* Close `phone_test.py` if it's running. Only one program can host the phone link.

## Settings: edit this cell

In [ ]:
# "ble" = real cars, "sim" = simulated cars (rehearsal at home)
LINK = "ble"

# The micro:bit names IN ROBOT ORDER: robot 0 first.
# Testing one car? Put its name first and keep the placeholders for the rest.
NAMES = ["CHANGEME", "none1", "none2"]

# Which rear marker colour each robot wears (for your notes / the report).
COLOURS = ["red", "green", "blue"]

## Setup: run once (no output expected except the last line)

In [ ]:
import asyncio, datetime, json, logging, sys, time
from pathlib import Path

if sys.platform == "win32":
    # Jupyter can leave the main thread in a COM mode that breaks bleak's callbacks.
    from bleak.backends.winrt.util import uninitialize_sta
    uninitialize_sta()

import links, protocol, link_test

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(name)s %(message)s",
                    datefmt="%H:%M:%S", force=True)
logging.getLogger("bleak").setLevel(logging.WARNING)

camera = [0]                     # phone packets received (step 7)
results = {"date": datetime.datetime.now().isoformat(timespec="seconds"), "tests": []}
link = None

def record(name, data):
    results["tests"].append({"test": name, "names": list(NAMES), "data": data})

def show_cars():
    if isinstance(link, links.BleDirectLink):
        for b in link.hub.bots:
            rid = link.robot_id(b)
            print(f"  {b.name:28s} robot {rid}  {'connected' if b.connected else 'DISCONNECTED'}")
        if not link.hub.bots:
            print("  no cars connected")
    elif isinstance(link, links.SimLink):
        print("  3 simulated cars")

async def connect():
    """(Re)connect with the current NAMES. Safe to run again."""
    global link
    if link is not None:
        await link.stop()
        link = None
        await asyncio.sleep(1.0)
    if LINK == "sim":
        link = links.SimLink(seed=1)
    else:
        link = links.BleDirectLink(NAMES, scan_seconds=10,
                                   on_camera=lambda _p: camera.__setitem__(0, camera[0] + 1))
    await link.start()
    show_cars()

print("setup OK, link =", LINK)

## Step 1: connect one car

Put the car's name first in `NAMES`, run the settings cell again, then run this cell. It scans for about 10 s.

**Expect:** `connected: bot 1 = BBC micro:bit [<name>]` and `<name> is robot 0`. The car's LEDs show `?`, then `0`.

**If you get `found 0 micro:bit(s)`:** check that the car shows the diamond, remove it from Windows Bluetooth settings, re-check *No Pairing Required*, and run the cell again.

In [ ]:
await connect()

## Step 2: ping latency

**Expect:** robot 0 gets numbers. Robots that are placeholders say *no replies*, which is expected.
One-way latency ≈ RTT ÷ 2.

In [ ]:
r = await link_test.ping_all(link, 30)
record("ping, one car", r)

## Step 3: motion and safety stop (wheels OFF the floor: stand the car on a cup)

**Expect:**
1. Robot 0 rotates **left** for about 1 s (left wheel backward, right wheel forward), then stops.
2. Both wheels briefly go forward, then **stop by themselves within about 0.3 s** (the watchdog).

**If the wheels turn the wrong way:** note which, and the firmware signs need swapping.
**If the car does not stop by itself, stop here** and don't put it on the floor.

In [ ]:
await link_test.drive_demo(link)

In [ ]:
# Record what you saw:
WHEELS_CORRECT = None      # True / False
STOPPED_BY_ITSELF = None   # True / False
record("drive + watchdog", {"wheels_correct": WHEELS_CORRECT, "stopped_by_itself": STOPPED_BY_ITSELF})

## Step 4: drop and reconnect

Run the cell. While robot 0 is being pinged (about the first 20 s), **switch the car off for about 3 s, then back on**.

**Expect:** `disconnected` → some lost pings → `reconnected` → `... is robot 0` → the car shows `0` again and pings continue.

In [ ]:
r = await link_test.ping_all(link, 400)
record("ping with power-cycle", r)
show_cars()

## Step 5: all three cars (all on cups)

1. Flash the other two cars with the **same** `.hex`, and label them with their names.
2. Choose robot 0, 1, 2 and stick on the matching rear colour (`COLOURS` above).
3. Put all three names in `NAMES` **in robot order**, then re-run the settings cell and this cell.

**Expect:** every car shows its own digit. In the drive test the cars spin **one at a time: robot 0, then 1, then 2**.
If the wrong car spins, the names are in the wrong order.

In [ ]:
await connect()

In [ ]:
r = await link_test.ping_all(link, 30)
record("ping, three cars", r)
await link_test.drive_demo(link)

## Step 6: speed calibration (one car ON THE FLOOR, about 1 m clear ahead)

The policy was trained in a simulator where cars drive **0.20 m/s** and spin **3 rad/s**.
These cells drive robot `CAL_ROBOT` and suggest new `FWD` / `SPIN` values for the firmware.

**6a.** Mark where the car starts, then run:

In [ ]:
CAL_ROBOT = 0
s = link_test.Sender(link)
acts = [0, 0, 0]; acts[CAL_ROBOT] = 1
await s.hold(acts, 2.0)                       # forward 2.0 s, then stop
print("measure how far it drove (target 40 cm)")

**6b.** Put it back and mark its heading, then run (it spins left for 2.1 s):

In [ ]:
acts = [0, 0, 0]; acts[CAL_ROBOT] = 2
await s.hold(acts, 2.1)
print("measure how far it turned (target 360 degrees = one full turn)")

**6c.** Type your measurements and the firmware's current values, then run:

In [ ]:
DIST_CM = None          # e.g. 33
TURN_DEG = None         # e.g. 300
FWD_NOW, SPIN_NOW = 40, 30   # what is in cutebot_ble.ts right now

if DIST_CM and TURN_DEG:
    fwd = round(FWD_NOW * 40 / DIST_CM)
    spin = round(SPIN_NOW * 360 / TURN_DEG)
    print(f"put these in firmware/cutebot_ble.ts, re-flash ALL cars, then repeat 6a-6c once:")
    print(f"  const FWD = {fwd}\n  const SPIN = {spin}\n  const REV = {-round(0.65 * fwd)}")
    record("calibration", {"robot": CAL_ROBOT, "fwd_now": FWD_NOW, "spin_now": SPIN_NOW,
                           "dist_cm": DIST_CM, "turn_deg": TURN_DEG, "fwd_new": fwd, "spin_new": spin})
else:
    print("fill in DIST_CM and TURN_DEG first")
print("If the car curved instead of driving straight, note which way: it needs a wheel trim.")

After re-flashing, run **Step 5's connect cell** again before continuing. Rebooted cars need their IDs re-sent.

## Step 7: latency under real load (3 cars + phone)

Run the cell, then on the phone: Bluefy → camera page → **Connect to laptop** → pick this PC.
It waits up to 3 minutes for camera packets, then pings all three cars while the phone streams.

**Expect:** about 12 camera packets/s and ping numbers similar to step 5. The *difference* from step 5 is the cost of the phone link.

In [ ]:
if LINK == "ble":
    await link_test.wait_for_phone(camera)
c0, t0 = camera[0], time.monotonic()
r = await link_test.ping_all(link, 100)
rate = (camera[0] - c0) / (time.monotonic() - t0)
print(f"camera packets during ping: {rate:.1f}/s (page sends ~12.5/s)")
record("ping, three cars + phone", {"ping": r, "camera_packets_per_s": round(rate, 1)})

## Step 8: results table, save, disconnect

Prints a Markdown table you can paste into the report, and saves everything to `results/`.

In [ ]:
rows = ["| Test | Robot | Colour | Median RTT (ms) | p95 (ms) | Lost | ≈ one-way (ms) |",
        "|---|---|---|---|---|---|---|"]
for t in results["tests"]:
    pings = t["data"].get("ping", t["data"]) if isinstance(t["data"], dict) else {}
    for rid, p in pings.items():
        if not isinstance(p, dict) or "median_ms" not in p:
            continue
        one_way = f"{p['median_ms'] / 2:.1f}" if p["median_ms"] is not None else "-"
        rows.append(f"| {t['test']} | {rid} | {COLOURS[int(rid)]} | {p['median_ms']} | "
                    f"{p['p95_ms']} | {p['lost']}/{p['sent']} | {one_way} |")
print("\n".join(rows))

Path("results").mkdir(exist_ok=True)
fname = Path("results") / f"link_test_{datetime.datetime.now():%Y%m%d_%H%M%S}_{LINK}.json"
fname.write_text(json.dumps(results, indent=2))
print("\nsaved", fname)

In [ ]:
if link is not None:
    await link.stop()      # sends STOP to every car, then disconnects
    link = None
print("disconnected")